# House Price Prediction — Data Cleaning & Exploration

## Objective

This notebook is the first step in the House Price Prediction project. Its
purpose is **exploration and documentation only** — getting familiar with
the raw California Housing dataset before any cleaning, preprocessing, or
modeling happens.

In this notebook we will:
- Load the raw dataset using the project's existing `src.data_loader` module.
- Look at its shape, a preview of the rows, and summary statistics.
- Check for missing values and duplicate rows.
- Review the data types of each column.
- Understand what each feature actually means.
- Run the project's existing `src.data_validation` checks and review the results.

**This notebook does NOT train any model, and does NOT build the
Streamlit dashboard** — those happen elsewhere in the project. No changes
are made to the data here; we are only looking at it.


## 1. Import Libraries

We need `pandas` for working with the data, plus a small path-setup step
so we can import the project's own `src` modules (`data_loader` and
`data_validation`) from inside this `notebooks/` folder.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

# This notebook lives in `notebooks/`, one level below the project root.
# Add the project root to the Python path so imports like
# `from src.data_loader import load_housing_data` work correctly no
# matter where Jupyter was launched from.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data_loader import load_housing_data
from src.data_validation import run_data_validation

print("Libraries imported successfully.")


## 2. Load the California Housing Dataset

We load the raw dataset using `load_housing_data()` from
`src/data_loader.py`. This function fetches the built-in scikit-learn
California Housing dataset and returns it as a single pandas DataFrame
containing the 8 feature columns plus the target column, `MedHouseVal`.

No cleaning happens here — this is the raw data, exactly as
`data_loader.py` provides it.


In [ ]:
housing_df = load_housing_data()
print(f"Dataset loaded: {housing_df.shape[0]} rows, {housing_df.shape[1]} columns.")


## 3. First Look at the Data

Before doing anything else, let's get a general feel for the dataset:
its shape, a preview of a few rows, column info, and summary statistics.


**Shape** — (number of rows, number of columns):

In [ ]:
housing_df.shape

**Preview** — the first 5 rows:

In [ ]:
housing_df.head()

**Info** — column names, non-null counts, and data types:

In [ ]:
housing_df.info()

**Describe** — summary statistics for every numeric column:

In [ ]:
housing_df.describe()

## 4. Analyze Missing Values

Next, we check whether any column has missing (NaN) values. This matters
because later preprocessing steps need to know whether — and how — to
handle them.


In [ ]:
missing_values = housing_df.isnull().sum()
print("Missing values per column:")
print(missing_values)

total_missing = int(missing_values.sum())
print(f"\nTotal missing values in the dataset: {total_missing}")


## 5. Analyze Duplicate Rows

We also check whether any rows are exact duplicates of each other.
Duplicate rows can quietly bias a model later on if they aren't
accounted for.


In [ ]:
num_duplicate_rows = int(housing_df.duplicated().sum())
print(f"Number of duplicate rows: {num_duplicate_rows}")


If any duplicates exist, here's a preview of them:

In [ ]:
if num_duplicate_rows > 0:
    display(housing_df[housing_df.duplicated(keep=False)].head(10))
else:
    print("No duplicate rows to display.")


## 6. Data Types

Let's look specifically at each column's data type. All feature columns
and the target should be numeric (`float64`), since scikit-learn's
California Housing dataset doesn't include any categorical columns.


In [ ]:
housing_df.dtypes

## 7. What Do These Features Mean?

The California Housing dataset describes housing in California,
aggregated at the **block group** level (a block group is the smallest
geographic unit the US Census Bureau publishes sample data for —
typically 600–3,000 people).

| Column | Meaning |
|---|---|
| `MedInc` | Median income for households in the block group, in **tens of thousands of dollars** (e.g. `3.5` means about $35,000). |
| `HouseAge` | Median age of houses in the block group, in **years**. |
| `AveRooms` | Average number of rooms per household in the block group. |
| `AveBedrms` | Average number of bedrooms per household in the block group. |
| `Population` | Total population of the block group. |
| `AveOccup` | Average number of household members (people per household). |
| `Latitude` | Geographic latitude of the block group. |
| `Longitude` | Geographic longitude of the block group. |
| `MedHouseVal` | **Target column.** Median house value for the block group, in **units of $100,000** (e.g. `2.5` means about $250,000). |

A couple of things worth keeping in mind for later steps:
- `AveRooms`, `AveBedrms`, and `AveOccup` are **averages per household**,
  not totals — a block group with very few households can produce
  unusually large or small averages, showing up as outliers later.
- `MedInc` and `MedHouseVal` are both capped in the original dataset
  (very high incomes/values are grouped into a single top bucket), which
  can show up as a cluster of identical maximum values.


## 8. Run the Project's Data Validation Checks

The project already has a dedicated validation module,
`src/data_validation.py`, which checks the DataFrame's structure
(correct columns, correct types, missing values, duplicates) **without
modifying anything**. Let's run its combined check,
`run_data_validation()`, against the raw data we just loaded.


In [ ]:
validation_report = run_data_validation(housing_df)
validation_report


## 9. Validation Results

The report above is a nested dictionary. Let's break it down piece by
piece so it's easy to read.


In [ ]:
print("Overall valid:", validation_report["overall_valid"])

print("\nStructure check:")
for key, value in validation_report["structure"].items():
    print(f"  {key}: {value}")

print("\nTotal missing values:", validation_report["total_missing_values"])
print("Duplicate rows:", validation_report["duplicate_rows"])

print("\nNumeric columns check:")
for key, value in validation_report["numeric_columns"].items():
    print(f"  {key}: {value}")


## Summary

- The dataset loaded successfully with the expected shape and columns.
- Missing values and duplicate rows were counted above but **not
  removed** — this notebook is for exploration and documentation only.
- The project's own validation checks confirm whether the DataFrame's
  structure is ready to move on to preprocessing.

**Next steps** (not part of this notebook): cleaning and preprocessing
happen in `src/preprocessing.py`, and model training happens in later
steps of the project.
